# Descriptive statistics, cohort and build checks

Feeds: **§4 Experiment Planning** (cohort, variant parity, control-build screen),
**§5 Experiment Execution** (filter chain, repeatability, selection funnel) and
**§6.1 Descriptive statistics**.

Independent of the other notebooks; run them in any order.
Every artifact written here is entered in `6_results_analysis/MANIFEST.csv` as **BODY**
(a table or figure in the thesis), **REPO** (cited by path) or **SENSITIVITY** (a sensitivity check the thesis does not cite).

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import thesis_lib as T

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
REF = T.reference()
for d in (T.TABLES, T.FIGURES, T.SENSITIVITY):
    d.mkdir(exist_ok=True)
T.figure_style()
NB = "notebooks/Descriptive_Statistics.ipynb"

## §5: Filter chain

Seven filters, applied in a fixed order. The final set does not depend on the order; only the intermediate counts do.
The *build-type parity* step (added 2026-10-02) removes itsok's Allatori build: it is a release build, and the protocol
committed before data collection (2026-07-27) requires every build to be a debug
build. It is rule enforcement on a property of the build, and touches no other arm.
Two filters remove nothing on this data (*scenarios complete* and *workload_joules > 0*), and the thesis states this rather than implying it.

In [2]:
raw = T.load_raw()
print(f"loaded: {len(raw)} rows")
df, chain = T.apply_filters(raw)
print(chain.to_string())
chain.to_csv(T.TABLES / "filter_chain.csv")
T.register("tables/filter_chain.csv", "REPO", "5 Execution",
           "cells surviving each step of the filter chain, per batch", NB)
ref = REF["filter_chain"]["series_pooled"]["pilot_regime_cutoff"]
T.check("series cells entering the pooled analysis", chain.loc[chain.index[-1], "series pooled"], ref, tol=0)
T.check("pilot cells after the regime cutoff", chain.loc[chain.index[-1], "pilot"], REF["filter_chain"]["pilot"]["pilot_regime_cutoff"], tol=0)
T.record("filter_chain.series_cells_analysed", value=int(chain.loc[chain.index[-1], "series pooled"]),
         loaded=int(chain.loc["loaded", "series pooled"]))
T.record("filter_chain.pilot_cells_after_regime_cutoff", value=int(chain.loc[chain.index[-1], "pilot"]))
series = df[df["batch"].isin(T.SERIES)].copy()

loaded: 4800 rows
batch                                                         pilot  series1  series2  series3  series pooled
loaded                                                         1200     1200     1200     1200           3600
scientifically_valid                                           1120     1112     1106     1113           3331
not a negative control                                         1070     1062     1057     1063           3182
cohort (drop androidskk)                                        950      942      938      944           2824
build type (drop itsok's release Allatori build)                921      914      908      914           2736
build parity (drop medtimer's packed build: coverage probes)    892      884      879      884           2647
scenarios complete                                              892      884      879      884           2647
workload_joules > 0                                             892      884      879      884        

### Sensitivity: the same chain on ten applications (not in the thesis)

Written to `tables/sensitivity_ten_apps/`. No ten-application figure appears in the thesis.

In [3]:
_, chain10 = T.apply_filters(raw, exclude_apps=())
chain10.to_csv(T.SENSITIVITY / "filter_chain_ten_apps.csv")
T.register("tables/sensitivity_ten_apps/filter_chain_ten_apps.csv", "SENSITIVITY", "none",
           "filter chain with androidskk kept (ten applications); not cited in the thesis", NB)
print(chain10[["pilot", "series pooled"]].to_string())

batch                                                         pilot  series pooled
loaded                                                         1200           3600
scientifically_valid                                           1120           3331
not a negative control                                         1070           3182
cohort (drop none)                                             1070           3182
build type (drop itsok's release Allatori build)               1041           3094
build parity (drop medtimer's packed build: coverage probes)   1012           3005
scenarios complete                                             1012           3005
workload_joules > 0                                            1012           3005
pilot regime (start >= 2026-08-07)                              740           3005


## §4: The cohort (Table 1)

Nine applications. androidskk was measured but is excluded from every arm: its control build carries obfuscated
code (Table 2), and all four builds of an application share one control.
An arm is marked analysed when the application contributes at least one matched pair to it.

In [4]:
chars = pd.read_csv(T.DATA / "inputs" / "app_characteristics.csv").set_index("app")
apps = sorted(series["app"].unique())
assert "androidskk" not in apps
# the workload: distinct scenarios that executed (control cells of the series), and scenario runs per cell
import re as _re
sj = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["app", "variant", "scientifically_valid", "scenarios_json",
                                                    "scenarios_planned"], dtype=str) for b in T.SERIES)
sj = sj[(sj["variant"] == "baseline_debug") & (sj["scientifically_valid"] == "True") & sj["scenarios_json"].notna()
        & sj["app"].isin(apps)]
def _names(js):
    # a scenario counts only if it ran (runs > 0): documenter lists two scenarios that never ran in any cell
    try:
        return {_re.sub(r"(_repeat\d+.*|_r\d+$|\s+round=\d+)", "", k)
                for k, v in json.loads(js).items() if v.get("runs", 0) > 0}
    except ValueError:
        return set()
def scen_table(d):
    return d.groupby("app").agg(distinct_scenarios=("scenarios_json", lambda c: len(set().union(*map(_names, c)))),
                                runs_per_cell=("scenarios_planned", lambda c: int(pd.to_numeric(c).mode().iloc[0])))
scen = scen_table(sj)
scen.to_csv(T.TABLES / "scenarios_per_app.csv")
T.register("tables/scenarios_per_app.csv", "REPO", "5 Execution (workload)",
           "distinct scenarios per application, and scenario runs per cell (some scenarios repeat to fill 120 s)", NB)
T.record("scenarios", distinct_total=int(scen["distinct_scenarios"].sum()), runs_per_cell_total=int(scen["runs_per_cell"].sum()),
         n_apps=len(scen))
print(scen.to_string()); print(f"total: {int(scen['distinct_scenarios'].sum())} distinct scenarios, "
                               f"{int(scen['runs_per_cell'].sum())} scenario runs per cell across the nine")
# not cited in the thesis: the same count with androidskk (ten applications)
sj10 = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["app", "variant", "scientifically_valid", "scenarios_json",
                                                      "scenarios_planned"], dtype=str) for b in T.SERIES)
sj10 = sj10[(sj10["variant"] == "baseline_debug") & (sj10["scientifically_valid"] == "True") & sj10["scenarios_json"].notna()]
scen10 = scen_table(sj10)
scen10.to_csv(T.SENSITIVITY / "scenarios_ten_apps.csv")
T.register("tables/sensitivity_ten_apps/scenarios_ten_apps.csv", "SENSITIVITY", "none",
           "scenario counts with androidskk; not cited in the thesis", NB)
print(f"(sensitivity) ten applications: {int(scen10['distinct_scenarios'].sum())} distinct, "
      f"{int(scen10['runs_per_cell'].sum())} runs per cell")
WHY_NOT = {("documenter", "allatori"): "no working build",
           ("keep_recipe", "allatori"): "not transformed",
           ("linkhub", "allatori"): "not transformed",
           ("itsok", "allatori"): "excluded: build type"}
rows = []
for app in apps:
    phones = sorted(series.loc[(series["app"] == app) & (series["variant"] == "baseline_debug"), "phone"].unique())
    row = {"App": app, "Category": chars.loc[app, "category"], "Version": chars.loc[app, "version_name"],
           "App classes": int(chars.loc[app, "app_classes"]),
           "Scenarios": int(scen.loc[app, "distinct_scenarios"]),
           "Devices": ", ".join(T.PHONE_DISPLAY[p].replace("Pixel ", "P") for p in phones)}
    for arm in T.ARMS:
        n = len(T.pairs(series[series["app"] == app], arm, "workload_joules"))
        n_passive = len(T.pairs(T.phase_set(series[series["app"] == app], "passive"), arm, "pre_workload_joules"))
        row[T.name(arm)] = (("yes" if n_passive else "active only") if n      # medtimer: the passive-phase rule
                            else WHY_NOT.get((app, T.short(arm)), "no pairs"))
    rows.append(row)
cohort = pd.DataFrame(rows)
print(cohort.to_string(index=False))
paths = T.write_tex(cohort, "table1_cohort",
    caption="The nine applications, the devices each was measured on, and the arms it contributes matched pairs to.",
    label="tab:cohort", colfmt="lllrrlccc", star=True, colsep="4pt",   # nine columns: spans both columns of the page
    notes=f"Scenarios: distinct Appium scenarios that ran, {int(scen['distinct_scenarios'].sum())} in all; "
          "some repeat within a cell to fill its 120 s. P3/P6/P9: Pixel 3/6/9. keep_recipe does not run on the Pixel 3. "
          "Active only: medtimer's passive phase contains its six-screen introduction, so medtimer is left out of the "
          "passive-phase analysis in every arm.",
    notes_tex=r"androidskk was measured and excluded from every arm, because its control build is already obfuscated. Allatori: "
              r"documenter's obfuscated build crashes at start-up, and keep\_recipe's and linkhub's were not transformed; itsok's is a "
              r"release build, excluded by the build-type rule (Section~\ref{sec:planning:variables}).")   # the note refers to no table outside the thesis body
T.register(paths, "APPENDIX", "4 Planning (subjects)", "cohort: application, category, size, scenarios, devices, arms analysed", NB)
T.record("cohort", n_apps=len(apps), apps=apps, excluded=["androidskk"],
         apps_per_arm={T.name(a): int(cohort[T.name(a)].isin(["yes", "active only"]).sum()) for a in T.ARMS},
         apps_per_arm_passive={T.name(a): int((cohort[T.name(a)] == "yes").sum()) for a in T.ARMS})

              distinct_scenarios  runs_per_cell
app                                            
calculator                     5              5
documenter                     5              8
itsok                         10             10
keep_recipe                    6             19
linkhub                        5              5
medtimer                      11             15
pdfviewer                      8             11
poetskingdom                   6              6
tipuous                        7              7
total: 63 distinct scenarios, 86 scenario runs per cell across the nine


(sensitivity) ten applications: 72 distinct, 104 runs per cell


         App                         Category      Version  App classes  Scenarios    Devices  Bangcle             Allatori    freeRASP
  calculator                       calculator        1.7.2          108          5 P3, P6, P9      yes                  yes         yes
  documenter      document viewer / converter 1.0.14.debug          189          5 P3, P6, P9      yes     no working build         yes
       itsok            mental-health journal        0.0.1          395         10 P3, P6, P9      yes excluded: build type         yes
 keep_recipe                   recipe manager          1.1          354          6     P6, P9      yes      not transformed         yes
     linkhub                 bookmark manager        2.1.0          852          5 P3, P6, P9      yes      not transformed         yes
    medtimer              medication reminder       1.17.7          729         11 P3, P6, P9 no pairs          active only active only
   pdfviewer                       PDF viewer   

## §4–§5: Workload duration, nine applications

How long the workload window lasted (`duration_workload_s`), over the analysed series cells: overall and per application,
and on the oldest device for the control alone, against the band the scenario lists were meant to hit there (100–130 s,
`appium_android_tests/CONVENTIONS.md` §6.5; a calibration that was specified but not carried out).

In [5]:
_wd_rows = []
for _wd_app, _wd_g in series.groupby("app"):
    _wd_p3 = _wd_g.loc[(_wd_g["phone"] == "pixel3") & (_wd_g["variant"] == "baseline_debug"), "duration_workload_s"]
    _wd_rows.append({"app": _wd_app, "cells": len(_wd_g), "median_workload_s": _wd_g["duration_workload_s"].median(),
                 "pixel3_control_cells": len(_wd_p3), "pixel3_control_median_s": _wd_p3.median() if len(_wd_p3) else float("nan"),
                 "pixel3_in_100_130_s": (100 <= _wd_p3.median() <= 130) if len(_wd_p3) else None})
_wd = pd.DataFrame(_wd_rows)
_wd_known = _wd.dropna(subset=["pixel3_control_median_s"])
_wd.loc[len(_wd)] = {"app": "all nine", "cells": len(series), "median_workload_s": series["duration_workload_s"].median(),
                   "pixel3_control_cells": int(_wd_known["pixel3_control_cells"].sum()), "pixel3_control_median_s": float("nan"),
                   "pixel3_in_100_130_s": f"{int(_wd_known['pixel3_in_100_130_s'].astype(bool).sum())} of {len(_wd_known)}"}
_wd.to_csv(T.TABLES / "workload_duration.csv", index=False)
T.register("tables/workload_duration.csv", "REPO", "4 Planning / 5 Execution (active-phase length)",
           "workload-window duration over the analysed series cells, overall and per application; Pixel 3 control medians against the 100-130 s band", NB)
print(_wd.round(1).to_string(index=False))
_wd_apps = _wd[_wd["app"] != "all nine"]
T.record("workload_duration", median_s=series["duration_workload_s"].median(),
         per_app_median_range_s=[_wd_apps["median_workload_s"].min(), _wd_apps["median_workload_s"].max()],
         pixel3_control_in_100_130_s=_wd.iloc[-1]["pixel3_in_100_130_s"],
         basis="analysed series cells (nine applications); Pixel 3 rows: the control's cells only")

         app  cells  median_workload_s  pixel3_control_cells  pixel3_control_median_s pixel3_in_100_130_s
  calculator    358               95.0                    30                     94.0               False
  documenter    267              101.0                    29                    101.0                True
       itsok    267              115.0                    29                    115.0                True
 keep_recipe    180              133.0                     0                      NaN                None
     linkhub    267              121.0                    29                    118.0                True
    medtimer    266              107.0                    28                    107.0                True
   pdfviewer    328              114.0                    23                    115.0                True
poetskingdom    356              108.0                    27                    108.0                True
     tipuous    358              126.0        

## §4: Variant parity (repository table; one sentence in the thesis)

Do the four builds of each application differ **only** by the protection? Input: `6_results_data/inputs/variant_parity.csv`,
measured from the exact APK each Android Runner config swept (`tools/verify_variant_parity.py`; see
`build_checks.PROVENANCE.md`). Checked per application:

- the four builds share `versionCode` and `versionName`;
- each APK's SHA-256 equals the hash the run tables recorded for its cells;
- each build carries its own protection's structural marker and no other (control and Allatori: none;
  Bangcle: the stub DEX; freeRASP: the Talsec start-up hook).

Two attributes differ and are disclosed in the last column; the provenance file explains why neither touches the measurement.

In [6]:
par = pd.read_csv(T.DATA / "inputs" / "variant_parity.csv")
par["short"] = par["variant"].map(T.short)
EXPECT = {"baseline": "NO_STRUCTURAL_MARKER", "allatori": "NO_STRUCTURAL_MARKER",
          "bangcle": "BANGCLE_STUB_DEX", "freerasp": "FREERASP_WIRED"}
par["marker_ok"] = par.apply(lambda r: r["structural_marker"] == EXPECT[r["short"]], axis=1)
rows = []
for app in apps:
    g = par[par["app"] == app].set_index("short")
    ctrl = g.loc["baseline"]
    diffs = []
    no_nsc = [T.DISPLAY[v] for v in g.index if g.loc[v, "network_security_config"] != ctrl["network_security_config"]]
    if no_nsc:
        diffs.append(("NSC absent: " if ctrl["network_security_config"] else "NSC added: ") + ", ".join(no_nsc))
    if g.loc["freerasp", "declares_internet"] and not ctrl["declares_internet"]:
        diffs.append("INTERNET added: freeRASP")
    not_dbg = [T.DISPLAY[v] for v in g.index if bool(g.loc[v, "debuggable"]) != bool(ctrl["debuggable"])]
    if not_dbg:
        diffs.append(("not a debug build: " if ctrl["debuggable"] else "debug build: ") + ", ".join(not_dbg))
    rows.append({"App": app, "Builds": len(g),
                 "Same version": "yes" if g["version_code"].nunique() == 1 and g["version_name"].nunique() == 1 else "NO",
                 "APK = run tables": "yes" if g["sha_matches_run_tables"].all() else "NO",
                 "Markers as expected": "yes" if g["marker_ok"].all() else "NO",
                 "Differences": "; ".join(diffs) or "none"})
parity = pd.DataFrame(rows)
print(parity.to_string(index=False))
bad = parity[(parity[["Same version", "APK = run tables", "Markers as expected"]] == "NO").any(axis=1)]
print(f"\napplications failing a parity check: {len(bad)}" + ("" if bad.empty else f"  <<< {list(bad['App'])}"))
parity.to_csv(T.TABLES / "variant_parity.csv", index=False)
T.register("tables/variant_parity.csv", "REPO", "4 Planning (variants; one sentence in the body)",
           "per application: same version, APK hash = run tables, protection markers, disclosed differences (NSC, INTERNET, debug build)", NB)
full = par[par["app"].isin(apps)].drop(columns=["short", "marker_ok"])
full.to_csv(T.TABLES / "variant_parity_full.csv", index=False)
T.register("tables/variant_parity_full.csv", "REPO", "4 Planning (variants)", "one row per build (36): version, SDKs, NSC, INTERNET, marker, hash", NB)

         App  Builds Same version APK = run tables Markers as expected                                                                          Differences
  calculator       4          yes              yes                 yes                              NSC absent: Bangcle, freeRASP; INTERNET added: freeRASP
  documenter       4          yes              yes                 yes                                                             INTERNET added: freeRASP
       itsok       4          yes              yes                 yes NSC absent: Bangcle, freeRASP; INTERNET added: freeRASP; not a debug build: Allatori
 keep_recipe       4          yes              yes                 yes                    NSC absent: Allatori, Bangcle, freeRASP; INTERNET added: freeRASP
     linkhub       4          yes              yes                 yes                              NSC absent: Bangcle, freeRASP; INTERNET added: freeRASP
    medtimer       4          yes              yes              

## §4 (Methods): The compilation state before each launch

Before every launch, the `before_run` hook asks Android to compile the application ahead of time
(`cmd package compile -f -m speed`) and saves what Android reports (`dumpsys package dexopt`).
`dexopt_status_series.csv` (`tools/extract_dexopt_status.py`; provenance beside it) holds, per cell, the filter requested
and the status Android reports for the application.

Android forces debuggable applications to verification only, whatever is requested (AOSP `PackageDexOptimizer`, and ART
Service's `Dexopter`: "We force vmSafeMode on debuggable apps as well"). Every analysed build is a debug build.

**The check that could fail:** the series contains one non-debuggable build, excluded from the analysis by the build-type
rule. If the request were honoured for every build, every row would read `speed`. If Android had not honoured it even there,
that row would read `verify` too.

In [7]:
key = ["batch", "phone", "app", "variant", "rep"]
dx = pd.read_csv(T.DATA / "inputs" / "dexopt_status_series.csv", dtype={"rep": str})
dx = dx.merge(full[["app", "variant", "debuggable"]].drop_duplicates(), on=["app", "variant"], how="inner",
              validate="many_to_one")                     # the nine applications' 36 checked builds (not androidskk)
assert dx["debuggable"].notna().all(), "a build without its debuggable flag"
isdbg = dx["debuggable"].astype(str).str.lower() == "true"
an = dx.merge(series[key], on=key, how="inner")
assert len(an) == len(series), "an analysed cell has no compilation record"
assert not an.merge(dx[~isdbg][key], on=key).shape[0], "a non-debuggable build among the analysed cells"


def counts(x):
    return {"cells": len(x), "requested_speed": int((x["requested_filter"] == "speed").sum()),
            "status_verify": int((x["status"] == "verify").sum()), "status_speed": int((x["status"] == "speed").sum())}


rows = [{"scope": "analysed cells", "group": T.name(v), **counts(an[an["variant"] == v])}
        for v in ("baseline_debug", "bangcle_debug", "allatori_debug", "freerasp_debug")]
nd = dx[~isdbg]
rows += [{"scope": "all series cells, nine applications", "group": "debuggable builds", **counts(dx[isdbg])},
         {"scope": "all series cells, nine applications", "group": "not debuggable (not analysed): " +
          ", ".join(sorted({f"{a} {T.name(v)}" for a, v in zip(nd["app"], nd["variant"])})), **counts(nd)}]
dxs = pd.DataFrame(rows)
dxs.to_csv(T.TABLES / "dexopt_status.csv", index=False)
T.register("tables/dexopt_status.csv", "REPO", "4 Planning / Methods (compilation before each launch)",
           "the application's compilation state before each launch: the filter requested (speed) and the status Android reports, "
           "analysed cells per technique and all series cells by build type (debuggable builds are only verified)", NB)
print(dxs.to_string(index=False))
a = dxs[dxs["scope"] == "analysed cells"]
T.record("dexopt.before_launch", requested="speed, in every cell",
         analysed_status_verify=f"{int(a['status_verify'].sum())} of {int(a['cells'].sum())}",
         not_debuggable_status_speed=f"{int(counts(nd)['status_speed'])} of {len(nd)}",
         basis="dumpsys package dexopt after cmd package compile -f -m speed, per cell (series)")

                              scope                                         group  cells  requested_speed  status_verify  status_speed
                     analysed cells                                       Control    760              760            760             0
                     analysed cells                                       Bangcle    677              677            677             0
                     analysed cells                                      Allatori    440              440            440             0
                     analysed cells                                      freeRASP    770              770            770             0
all series cells, nine applications                             debuggable builds   3145             3145           3145             0
all series cells, nine applications not debuggable (not analysed): itsok Allatori     90               90              0            90


## §4: Control-build screen (Table 2)

**Detection rule.** For each class in the application's own package (third-party libraries are not counted), read the
`SourceFile` attribute the compiler wrote into the DEX. An unobfuscated debug build keeps names such as `MainActivity.kt`;
Allatori rewrites them to one or two characters. *Real* = matches `Name.kt` / `Name.java`; *1–2 char* = the obfuscation signature.
**Classes with no `SourceFile` attribute are counted in neither direction**: missing debug information is not evidence of
obfuscation, and not evidence against it. Verdict: *fail* below 25% real names among classes that carry one; *warn* if any
1–2 character name is present; *pass* otherwise; and *unproven*, never pass, when fewer than 10 classes carry a name at
all. The minimum comes from the Allatori builds themselves (next cell): in every working one, a large share of named
classes carries the 1–2 character signature, so a control with ten or more named classes and none of them short is
strong evidence of no obfuscation, and a control with fewer is not evidence either way.

**Bangcle builds cannot be screened**: the packer encrypts the DEX, so the class table is not readable from the APK.
Input: `6_results_data/inputs/control_build_screen.csv` (`tools/verify_baseline_not_obfuscated.py`).

In [8]:
scr = pd.read_csv(T.DATA / "inputs" / "control_build_screen.csv")
# the screened APK is the APK that ran: compare with every hash the run tables recorded
rec = pd.concat(pd.read_csv(p, usecols=["app", "variant", "apk_sha256"], dtype=str) for p in T.RUN_TABLES.values())
rec = rec[(rec["variant"] == "baseline_debug") & rec["apk_sha256"].notna()].groupby("app")["apk_sha256"].agg(set)
scr["hash_matches_run_tables"] = scr.apply(lambda r: rec.get(r["app"], set()) == {r["apk_sha256"]}, axis=1)
print("screened APK = swept APK for all ten controls:", bool(scr["hash_matches_run_tables"].all()))
scr["named"] = scr["real_source_names"] + scr["short_source_names"] + scr["other_source_names"]
tab = pd.DataFrame({
    "App": scr["app"],
    "Real / named": scr["real_source_names"].astype(str) + " / " + scr["named"].astype(str),
    "1-2 char": scr["short_source_names"],
    "No attribute": scr["no_source_attribute"],
    "Verdict": scr["verdict"].str.lower(),
    "Cohort": np.where(scr["app"].isin(apps), "in", "excluded"),
})
print(tab.to_string(index=False))
al = pd.read_csv(T.DATA / "inputs" / "allatori_build_screen.csv").set_index("app")
al["named"] = al["real_source_names"] + al["short_source_names"] + al["other_source_names"]
al["short_share_pct"] = 100 * al["short_source_names"] / al["named"]
used = al.loc[[a for a in apps if a in al.index and al.loc[a, "short_source_names"] > 0]]
print(f"\nthe signature in the working Allatori builds: {used['short_share_pct'].min():.0f}-{used['short_share_pct'].max():.0f}% "
      f"of named classes; e.g. pdfviewer's Allatori build {int(al.loc['pdfviewer', 'short_source_names'])} of "
      f"{int(al.loc['pdfviewer', 'named'])}, its control 0 of {int(scr.set_index('app').loc['pdfviewer', 'named'])}")
paths = T.write_tex(tab, "table2_control_build_screen",
    caption="Control-build screen: compiler-written source-file names in each control APK.",
    label="tab:control-screen", colfmt="lrrrll", star=True,
    notes="Named: classes that carry a source-file attribute. Classes with no attribute are counted in neither direction; "
          "fewer than 10 named classes gives no verdict. Bangcle builds cannot be screened (encrypted DEX).")
T.register(paths, "SUPPLEMENT", "Supplement (validity of the control)", "per-app control screen, all ten measured applications incl. androidskk", NB)
T.record("control_screen", fail=list(scr.loc[scr["verdict"] == "FAIL", "app"]),
         warn=list(scr.loc[scr["verdict"] == "WARN", "app"]), passed=int((scr["verdict"] == "PASS").sum()),
         androidskk_real_over_named=tab.loc[tab["App"] == "androidskk", "Real / named"].iloc[0],
         linkhub_short_names=int(scr.loc[scr["app"] == "linkhub", "short_source_names"].iloc[0]))

screened APK = swept APK for all ten controls: True
         App Real / named  1-2 char  No attribute Verdict   Cohort
  androidskk      0 / 236       158            14    fail excluded
  calculator      91 / 98         0            10    pass       in
  documenter    158 / 177         0            12    pass       in
       itsok    329 / 378         0            17    pass       in
 keep_recipe    231 / 341         0            13    pass       in
     linkhub    451 / 830        44            22    warn       in
    medtimer    338 / 714         0            15    pass       in
   pdfviewer      18 / 33         0            18    pass       in
poetskingdom    497 / 594         0            14    pass       in
     tipuous     82 / 213         0            26    pass       in

the signature in the working Allatori builds: 41-93% of named classes; e.g. pdfviewer's Allatori build 18 of 33, its control 0 of 33


### Note: where linkhub's 1–2 character classes live

Computed from the `short_name_packages` column of the screen, and from the same screen run on the freeRASP builds
(`freerasp_build_screen.csv`).

In [9]:
def pk(s):
    return {} if pd.isna(s) or not s else {k: int(v) for k, v in (x.rsplit(":", 1) for x in s.split(";"))}
fr = pd.read_csv(T.DATA / "inputs" / "freerasp_build_screen.csv").set_index("app")
for app in ("linkhub", "androidskk"):
    r = scr.set_index("app").loc[app]
    pkgs = pk(r["short_name_packages"])
    root = chars.loc[app, "package"]
    own = sum(v for k, v in pkgs.items() if k == root or k.startswith(root + "."))
    print(f"{app}: {r['short_source_names']} classes with 1-2 char source names, {own} of them inside the app's own "
          f"namespace {root}; {len(pkgs)} packages: {', '.join(sorted(pkgs))}")
    print(f"   its freeRASP build: {fr.loc[app, 'short_source_names']} such classes, verdict {fr.loc[app, 'verdict'].lower()}")

linkhub: 44 classes with 1-2 char source names, 44 of them inside the app's own namespace com.amrdeveloper.linkhub; 10 packages: com.amrdeveloper.linkhub, com.amrdeveloper.linkhub.c, com.amrdeveloper.linkhub.f, com.amrdeveloper.linkhub.k, com.amrdeveloper.linkhub.ui, com.amrdeveloper.linkhub.ui.explorer, com.amrdeveloper.linkhub.ui.folder, com.amrdeveloper.linkhub.ui.importexport, com.amrdeveloper.linkhub.ui.link, com.amrdeveloper.linkhub.v
   its freeRASP build: 0 such classes, verdict pass
androidskk: 158 classes with 1-2 char source names, 158 of them inside the app's own namespace jp.deadend.noname.skk; 3 packages: jp.deadend.noname.skk, jp.deadend.noname.skk.engine, jp.deadend.noname.skk.p
   its freeRASP build: 0 such classes, verdict pass


**Finding.** linkhub's 44 classes are not a bundled pre-minified library: the screen only counts classes in the application's
own package, and all 44 sit in linkhub's own namespace: its UI subpackages and single-letter packages (`c`, `f`, `k`, `v`)
of the kind Allatori's package renaming produces. linkhub's freeRASP build carries none, so the residue is in the control only.
androidskk shows the same pattern at full scale (all 158 inside `jp.deadend.noname.skk`, 0 real names left), where the cause is
established: a stale Allatori output shipped as the control. For linkhub that is the likely mechanism but **not proven**, and
whether the 44 classes are ever loaded at run time is **not established**. Its own code is intact (451 real names), so the residue
is extra code next to the real application, not a replacement of it. Threats to validity, with the leave-linkhub-out
sensitivity in `tables/linkhub_leave_out.csv` (Headline_Effects).

## §6.1: Descriptive statistics per technique

Computed at the cell level over the pooled series, per technique and energy window:
mean, standard deviation (n−1), median, minimum, maximum and coefficient of variation (100·s/x̄).
The full application × technique × device grid is written to `tables/` for the appendix.

In [10]:
rows = []
for var in ("baseline_debug", "allatori_debug", "bangcle_debug", "freerasp_debug"):
    for w, col in T.WINDOWS.items():
        s_ = T.phase_set(series, w)                    # the passive phase leaves out medtimer
        x = s_.loc[(s_["variant"] == var) & (s_[col] > 0), col]
        rows.append(dict(technique=T.name(var), window=w, n=len(x), mean=x.mean(),
                         sd=x.std(ddof=1), median=x.median(), min=x.min(), max=x.max(),
                         cv_pct=100 * x.std(ddof=1) / x.mean()))
desc = pd.DataFrame(rows)
desc.to_csv(T.TABLES / "descriptives_per_technique.csv", index=False)
T.register("tables/descriptives_per_technique.csv", "REPO", "6.1 Descriptives", "cell-level energy per technique and window: n, mean, sd, median, min, max, CV", NB)
print(desc.round(2).to_string(index=False))
KEY = {v: k for k, v in T.DISPLAY.items()}
for r in desc.itertuples():
    T.check(f"{r.technique} {r.window} mean (J)", r.mean,
            REF["descriptives_series_per_variant"][f"{KEY[r.technique]}_{r.window}"]["mean"], tol=0.01)

technique   window   n   mean    sd  median    min    max  cv_pct
  Control workload 760 319.07 77.30  304.88 179.15 723.75   24.23
  Control  startup 673  77.55 21.93   79.40  34.21 152.32   28.28
 Allatori workload 440 302.40 67.76  293.22 188.39 530.59   22.41
 Allatori  startup 351  80.58 18.87   83.56  43.01 147.27   23.42
  Bangcle workload 677 325.44 72.21  315.40 202.15 642.49   22.19
  Bangcle  startup 677  77.28 21.49   79.28  35.96 168.95   27.81
 freeRASP workload 770 329.10 70.09  320.15 189.24 751.33   21.30
 freeRASP  startup 680  99.82 21.79   98.10  47.94 192.50   21.83
  Control workload mean (J): +319.0651   (expected +319.0650)  OK
  Control startup mean (J): +77.5458   (expected +77.5460)  OK
  Allatori workload mean (J): +302.3974   (expected +302.3970)  OK
  Allatori startup mean (J): +80.5795   (expected +80.5790)  OK
  Bangcle workload mean (J): +325.4414   (expected +325.4410)  OK
  Bangcle startup mean (J): +77.2829   (expected +77.2830)  OK
  freeRASP worklo

## Table 3: energy per cell, what one measurement is

Per technique (all devices), and for the unprotected control per device. **CV** is taken over all cells of the group, so it
mixes applications and devices; **within-group CV** is the median, over (application, device) groups of at least three
cells, of the CV among repeated cells of the same application, device and technique: the measurement's repeatability.
The means are **not** comparable across techniques: the Allatori rows cover five applications and keep_recipe has no
Pixel 3 cells. The techniques are compared through matched pairs in Table 4.

In [11]:
def within_cv(d, col):
    g = d[d[col] > 0].groupby(["app", "phone"])[col].agg(["count", "mean", lambda x: x.std(ddof=1)])
    g.columns = ["n", "mean", "sd"]
    g = g[g["n"] >= 3]
    return float((100 * g["sd"] / g["mean"]).median())
rows = []
def add(name, d, key):
    for w, col in T.WINDOWS.items():
        d_ = T.phase_set(d, w)                          # the passive phase leaves out medtimer
        x = d_.loc[d_[col] > 0, col]
        r = {"Group": f"{name} ({d_['app'].nunique()} apps)", "Window": T.WINDOW_DISPLAY[w], "Cells": len(x),
             "Mean (J)": x.mean(), "Median (J)": x.median(), "SD (J)": x.std(ddof=1), "CV (%)": 100 * x.std(ddof=1) / x.mean(),
             "Within-group CV (%)": within_cv(d_, col)}
        rows.append(r)
        T.record(f"descriptives.{key}.{w}", cells=len(x), mean_j=r["Mean (J)"], median_j=r["Median (J)"], sd_j=r["SD (J)"],
                 cv_pct=r["CV (%)"], within_group_cv_pct=r["Within-group CV (%)"], n_apps=int(d_["app"].nunique()))
for var in ("baseline_debug", "bangcle_debug", "allatori_debug", "freerasp_debug"):
    add(T.name(var), series[series["variant"] == var], T.short(var))
n_tech = len(rows)
ctrl = series[series["variant"] == "baseline_debug"]
for ph in ("pixel3", "pixel6", "pixel9"):
    add(T.PHONE_DISPLAY[ph], ctrl[ctrl["phone"] == ph], f"control_{ph}")
t3 = pd.DataFrame(rows)
print(t3.round(1).to_string(index=False))
shown = t3.copy()
for c_ in ("Mean (J)", "Median (J)", "SD (J)", "Within-group CV (%)"):
    shown[c_] = shown[c_].map(lambda v: f"{v:.1f}")
shown["CV (%)"] = shown["CV (%)"].map(lambda v: f"{v:.0f}")
paths = T.write_tex(shown, "table3_descriptives",
    caption="Energy per cell, by technique and, for the unprotected control, by device (three series batches).",
    label="tab:descriptives", colfmt="llrrrrrr",
    blocks={0: "By technique, all devices", n_tech: "Unprotected control, by device"},
    notes="Passive phase: medtimer left out in every build. CV: over all cells of the group (mixes applications and devices). Within-group CV: median CV among repeated "
          "cells of one application, device and technique. Means are not comparable across techniques (different "
          "application sets);",
        notes_tex=r"compare techniques through Table~\ref{tab:main-results}.")
T.register(paths, "REPO", "6.1 Descriptives (pooled; the thesis uses the per-device table)",
           "energy per cell per technique and, for the control, per device: cells, mean, median, SD, CV, within-group CV", NB)

            Group  Window  Cells  Mean (J)  Median (J)  SD (J)  CV (%)  Within-group CV (%)
 Control (9 apps)  Active    760     319.1       304.9    77.3    24.2                  5.2
 Control (8 apps) Passive    673      77.5        79.4    21.9    28.3                  6.0
 Bangcle (8 apps)  Active    677     325.4       315.4    72.2    22.2                  4.7
 Bangcle (8 apps) Passive    677      77.3        79.3    21.5    27.8                  6.2
Allatori (5 apps)  Active    440     302.4       293.2    67.8    22.4                  4.8
Allatori (4 apps) Passive    351      80.6        83.6    18.9    23.4                  6.1
freeRASP (9 apps)  Active    770     329.1       320.2    70.1    21.3                  6.5
freeRASP (8 apps) Passive    680      99.8        98.1    21.8    21.8                  8.5
 Pixel 3 (8 apps)  Active    224     263.1       240.5    59.8    22.7                  3.7
 Pixel 3 (7 apps) Passive    196      63.7        58.0    18.3    28.8          

In [12]:
grid = (series.melt(id_vars=["app", "variant", "phone"], value_vars=list(T.WINDOWS.values()),
                    var_name="measure", value_name="joules")
        .query("joules > 0")
        .groupby(["app", "variant", "phone", "measure"])["joules"]
        .agg(n="count", mean="mean", sd=lambda s: s.std(ddof=1), median="median", min="min", max="max"))
grid["cv_pct"] = 100 * grid["sd"] / grid["mean"]
grid = grid.reset_index()
grid["variant"] = grid["variant"].map(T.name)
grid.rename(columns={"variant": "technique"}).to_csv(T.TABLES / "descriptives_app_technique_device.csv", index=False)
T.register("tables/descriptives_app_technique_device.csv", "REPO", "Appendix", "descriptives per application x technique x device x window", NB)
print(f"appendix grid: {len(grid)} rows written")

appendix grid: 180 rows written


## §5: Repeatability, pilot versus series

**Same CV formula, different grouping** from the descriptive table: groups are (app, phone, technique) **without batch**,
n ≥ 3 cells, and the figure reported is the **median CV across the groups present in both regimes**.
Quoted as a direction and magnitude, not as a p-value.

In [13]:
pilot = df[df["batch"] == "pilot"]
series_w = series.assign(watts=series["workload_avg_watts"])
pilot_w = pilot.assign(watts=pilot["workload_avg_watts"])

def cvs(d, col):
    g = d[d[col] > 0].groupby(["app", "phone", "variant"])[col]
    s = g.agg(["count", "mean", lambda x: x.std(ddof=1)])
    s.columns = ["n", "mean", "sd"]
    s = s[s["n"] >= 3]
    return 100 * s["sd"] / s["mean"]

rows = []
for label, col in (("workload_energy", "workload_joules"), ("startup_energy", "pre_workload_joules"),
                   ("workload_duration", "duration_workload_s"), ("mean_power", "watts")):
    _ps = (lambda d: T.phase_set(d, "passive")) if col == "pre_workload_joules" else (lambda d: d)   # medtimer's passive-phase rule
    cp, cs = cvs(_ps(pilot_w), col), cvs(_ps(series_w), col)
    common = cp.index.intersection(cs.index)
    rows.append(dict(measure=label, groups_common=len(common),
                     pilot_median_cv=cp[common].median(), series_median_cv=cs[common].median(),
                     tighter_in_series=int((cs[common] < cp[common]).sum()),
                     pilot_cv_gt_15=f"{int((cp > 15).sum())}/{len(cp)}",
                     series_cv_gt_15=f"{int((cs > 15).sum())}/{len(cs)}"))
rep = pd.DataFrame(rows)
rep.to_csv(T.TABLES / "repeatability_cv.csv", index=False)
T.register("tables/repeatability_cv.csv", "REPO", "5 Execution", "median within-group CV, pilot vs series, four measures", NB)
print(rep.round(2).to_string(index=False))
for r in rep.itertuples():
    T.check(f"{r.measure} series median CV", r.series_median_cv, REF["repeatability"][r.measure]["series_median_cv"], tol=0.01)
w = rep.set_index("measure").loc["workload_energy"]
T.record("repeatability.workload_energy_median_cv", pilot_pct=w.pilot_median_cv, series_pct=w.series_median_cv,
         groups=int(w.groups_common), cv_over_15_pilot=w.pilot_cv_gt_15, cv_over_15_series=w.series_cv_gt_15)

          measure  groups_common  pilot_median_cv  series_median_cv  tighter_in_series pilot_cv_gt_15 series_cv_gt_15
  workload_energy             88            10.00              5.53                 61          30/88           11/90
   startup_energy             79             8.60              6.39                 56          29/79           20/81
workload_duration             88             3.65              1.58                 80           0/88            0/90
       mean_power             88             9.66              5.76                 62          31/88           14/90
  workload_energy series median CV: +5.5343   (expected +5.5340)  OK
  startup_energy series median CV: +6.3874   (expected +6.3870)  OK
  workload_duration series median CV: +1.5782   (expected +1.5780)  OK
  mean_power series median CV: +5.7569   (expected +5.7570)  OK


## Threats: why the Pixel 9 is the least precise device (power versus arrival temperature)

Series batches only. Before every cell, after the 60 s settle and the maintenance guard, the thermal gate read the
phone's skin temperature, then rested the phone if it was moderately throttled or worse (at most 10 min); the
measurement followed (`tools/sweep_runner_parallel_ykush.sh` runs the settle, then the gate). The **arrival temperature** is that first reading, before any rest. The pilot ran without the gate. A reading of 0 is a
failed read; it is left out of the temperature columns (`thesis_lib.load_raw`), and those cells stay in every energy
figure.

Everything is **within a group** (application, device, technique, batch), so differences between applications,
techniques and batches cannot enter:

* a cell's **power deviation** is its mean workload power relative to its group's mean, in %;
* its **arrival-temperature deviation** is its arrival temperature minus its group's mean, in °C;
* its **duration deviation** is its workload duration relative to its group's mean, in %.

Per device: the Pearson correlation of power deviation with arrival-temperature deviation, and the least-squares
slope (% of power per °C). The slope is computed twice: within groups, and within groups **and** within the same
three-hour block of the same day (UTC), with fixed effects for both, so that a shared condition of the rig at that
hour cannot produce it. Then the spread of power and of duration, and the same deviations for post-charge cells
against the other cells of their groups.

**Balance.** For each matched pair, the treated group's mean arrival temperature minus the control group's,
averaged over pairs with a paired t-interval. Because the order of cells was randomised, any difference is chance.
Its consequence is checked rather than assumed: the difference times the device's within-group slope is the change
in ΔE that the temperature difference alone would produce (**implied shift**, percentage points), per device and
over all pairs.

In [14]:
G = ["app", "phone", "variant", "batch"]
d = series.copy()
d["power_dev_pct"] = 100 * (d["workload_avg_watts"] / d.groupby(G)["workload_avg_watts"].transform("mean") - 1)
d["duration_dev_pct"] = 100 * (d["duration_workload_s"] / d.groupby(G)["duration_workload_s"].transform("mean") - 1)
d["arrival_temp_dev_c"] = d["arrival_temp_c"] - d.groupby(G)["arrival_temp_c"].transform("mean")
d["post_charge"] = T.post_charge_flag(d).to_numpy()
start = pd.to_datetime(d["cell_start_ts_utc"])
d["block"] = start.dt.strftime("%Y-%m-%d") + "_" + (start.dt.hour // 3).astype(str)
d["group"] = d[G].astype(str).agg("|".join, axis=1)
print("failed arrival-temperature reads (0), per device:",
      series.assign(failed=series["arrival_temp_c"].isna()).groupby("phone")["failed"].sum().to_dict())


def slope_with_fixed_effects(v, effects):
    """Least-squares slope of power deviation on arrival temperature, with a dummy per level of each effect."""
    X = np.column_stack([v["arrival_temp_c"].to_numpy(float)] +
                        [pd.get_dummies(v[e]).to_numpy(float) for e in effects])
    return float(np.linalg.lstsq(X, v["power_dev_pct"].to_numpy(float), rcond=None)[0][0])


rows, slope = [], {}
for ph, s in d.groupby("phone"):
    v = s.dropna(subset=["arrival_temp_dev_c"])
    slope[ph] = float(np.polyfit(v["arrival_temp_dev_c"], v["power_dev_pct"], 1)[0])
    assert abs(slope_with_fixed_effects(v, ["group"]) - slope[ph]) < 1e-6   # two routes to the within-group slope agree
    pc, pv = s["post_charge"], v["post_charge"]
    rows.append({"device": T.PHONE_DISPLAY[ph], "cells": len(s), "cells_with_arrival_temperature": len(v),
                 "r_power_vs_arrival_temperature": v["power_dev_pct"].corr(v["arrival_temp_dev_c"]),
                 "power_change_pct_per_degC": slope[ph],
                 "power_change_pct_per_degC_same_3h_block": slope_with_fixed_effects(v, ["group", "block"]),
                 "sd_power_dev_pct": s["power_dev_pct"].std(), "sd_duration_dev_pct": s["duration_dev_pct"].std(),
                 "post_charge_cells": int(pc.sum()),
                 "post_charge_arrival_temp_diff_degC": v.loc[pv, "arrival_temp_dev_c"].mean() - v.loc[~pv, "arrival_temp_dev_c"].mean(),
                 "post_charge_power_diff_pct": s.loc[pc, "power_dev_pct"].mean() - s.loc[~pc, "power_dev_pct"].mean()})
dev = pd.DataFrame(rows)
assert dev["cells"].sum() == len(series), "every series cell is in exactly one device row"
dev.to_csv(T.TABLES / "device_power_vs_arrival_temperature.csv", index=False)
T.register("tables/device_power_vs_arrival_temperature.csv", "REPO", "Threats (measurement precision)",
           "per device, within groups: power vs arrival temperature (r; % per degC, also within the same 3-h block); spread of power and duration; post-charge cells", NB)
print(dev.round(2).to_string(index=False))
for r in dev.itertuples():
    T.record(f"device_temperature.{r.device.replace(' ', '').lower()}", r_power_vs_arrival_temperature=r.r_power_vs_arrival_temperature,
             power_change_pct_per_degC=r.power_change_pct_per_degC,
             power_change_pct_per_degC_same_3h_block=r.power_change_pct_per_degC_same_3h_block,
             sd_power_dev_pct=r.sd_power_dev_pct, sd_duration_dev_pct=r.sd_duration_dev_pct,
             post_charge_arrival_temp_diff_degC=r.post_charge_arrival_temp_diff_degC, post_charge_power_diff_pct=r.post_charge_power_diff_pct,
             cells=r.cells, cells_with_arrival_temperature=r.cells_with_arrival_temperature)

g = d.groupby(["app", "phone", "batch", "variant"])["arrival_temp_c"].mean().reset_index()
ctrl = g[g["variant"] == "baseline_debug"]
bal = []
for arm in T.ARMS:
    pr = T.pairs(series, arm, "workload_joules")[["app", "phone", "batch"]]          # exactly the headline's pairs
    t = (pr.merge(g[g["variant"] == arm], on=["app", "phone", "batch"])
           .merge(ctrl, on=["app", "phone", "batch"], suffixes=("", "_control")))
    t["diff"] = t["arrival_temp_c"] - t["arrival_temp_c_control"]
    assert len(t) == len(pr) and t["diff"].notna().all(), "a headline pair has no arrival temperature on one side"
    t["shift"] = t["diff"] * t["phone"].map(slope)
    for ph, s in list(t.groupby("phone")) + [("all", t)]:
        m, lo, hi, _ = T.ci_naive(s["diff"])
        bal.append({"technique": T.name(arm), "device": T.PHONE_DISPLAY.get(ph, "all devices"), "pairs": len(s),
                    "arrival_temp_diff_degC": m, "ci_lo": lo, "ci_hi": hi, "implied_shift_pp": s["shift"].mean()})
bal = pd.DataFrame(bal)
bal.to_csv(T.TABLES / "arrival_temperature_balance.csv", index=False)
T.register("tables/arrival_temperature_balance.csv", "REPO", "Threats (measurement precision)",
           "arrival temperature, each technique minus its control, per device and over all pairs, with the implied shift in delta-E", NB)
print(bal.round(2).to_string(index=False))
per_dev = bal[bal["device"] != "all devices"]
pooled = bal[bal["device"] == "all devices"]
T.record("arrival_temperature_balance", diff_range_degC_per_device=[per_dev["arrival_temp_diff_degC"].min(), per_dev["arrival_temp_diff_degC"].max()],
         intervals_excluding_zero_per_device=int(((per_dev["ci_lo"] > 0) | (per_dev["ci_hi"] < 0)).sum()), intervals_per_device=len(per_dev),
         diff_range_degC_all_devices=[pooled["arrival_temp_diff_degC"].min(), pooled["arrival_temp_diff_degC"].max()],
         intervals_excluding_zero_all_devices=int(((pooled["ci_lo"] > 0) | (pooled["ci_hi"] < 0)).sum()),
         max_abs_implied_shift_pp_all_devices=pooled["implied_shift_pp"].abs().max())

failed arrival-temperature reads (0), per device: {'pixel3': 0, 'pixel6': 28, 'pixel9': 0}


 device  cells  cells_with_arrival_temperature  r_power_vs_arrival_temperature  power_change_pct_per_degC  power_change_pct_per_degC_same_3h_block  sd_power_dev_pct  sd_duration_dev_pct  post_charge_cells  post_charge_arrival_temp_diff_degC  post_charge_power_diff_pct
Pixel 3    796                             796                           -0.25                      -0.42                                    -0.55              3.01                 2.81                110                               -1.94                        1.43
Pixel 6    926                             898                           -0.67                      -1.76                                    -1.14              5.41                 2.78                190                               -1.95                        1.95
Pixel 9    925                             925                           -0.84                      -5.57                                    -6.49             13.45                 2.99        

## §4: Selection funnel (repository figure)

Rendered from `6_results_data/inputs/selection_funnel.csv`, which records the source of every count.

In [15]:
fun = pd.read_csv(T.DATA / "inputs" / "selection_funnel.csv")
fig, ax = plt.subplots(figsize=(T.FIG_SINGLE, 2.2), layout="constrained")
y = np.arange(len(fun))[::-1]
ax.barh(y, fun["apps"], color=T.COLOURS["Allatori"], height=0.65)
ax.set_xscale("log")
ax.set_yticks(y); ax.set_yticklabels(fun["stage"])
for yi, v in zip(y, fun["apps"]):
    ax.text(v * 1.1, yi, str(v), va="center")
ax.set_xlabel("Applications (log scale)")
ax.set_xlim(right=fun["apps"].max() * 4)
ax.spines[["top", "right"]].set_visible(False)
T.register(T.save_figure(fig, "selection_funnel", T.FIG_SINGLE), "REPO", "4 Planning (subjects)", "selection funnel from the clone tree to the nine-application cohort", NB)
print(fun.to_string(index=False))

                                stage  apps                                                                             source
                    AndroT clone tree  1324                AndroT dataset of Muradyan and Fazzini (MSR 2026): all repositories
                 Connected-test batch   172                                                   connected-test run of 2026-04-08
                        final_dataset    25 connected-test run of 2026-04-08: apps with a passing Espresso test on the Pixel 9
            Appium scenarios authored    22                                            appium_android_tests/<app>/scenarios.py
Four-variant cohort locked 2026-07-25    11                                                      experiment plan of 2026-07-25
  Measured in the four thesis batches    10                                                                the four run tables
   Final cohort (androidskk excluded)     9                                decision 2026-09-28; nine-applicatio

In [16]:
# Table 6 of the thesis: excluded cells by cause in the three series batches the estimates use; the pilot in the note.
_ex_cause = {"DEVICE_UNSUITABLE": "Device unsuitable", "APP_CRASHED": "Application crashed", "SCENARIO_FAILED": "Scenario failed",
             "HUB_RACE_CONTAMINATED": "Supply contamination", "HUB_ABORTED": "Harness fault", "ZOMBIE_NO_RUN_ID": "Harness fault",
             "AR_INTERNAL": "Harness fault"}
_ex_desc = {"Device unsuitable": "one application–device pair never completed", "Application crashed": "crashed repeatedly within the cell",
            "Scenario failed": "a scenario did not complete", "Supply contamination": "power reached the device mid-cell",
            "Harness fault": "runner or power-control failure"}
_ex = pd.concat(pd.read_csv(T.RUN_TABLES[b], usecols=["scientifically_valid", "cell_failure_class"], dtype=str).assign(batch=b)
                for b in ("pilot",) + T.SERIES)
_ex_bad = _ex[_ex["scientifically_valid"] != "True"]
assert not set(_ex_bad["cell_failure_class"]) - set(_ex_cause), set(_ex_bad["cell_failure_class"]) - set(_ex_cause)
_ex_bad = _ex_bad.assign(cause=_ex_bad["cell_failure_class"].map(_ex_cause))
_ex_n = _ex_bad.groupby(["cause", "batch"]).size().unstack(fill_value=0).reindex(list(_ex_desc), fill_value=0)
_ex_lab = dict(zip(T.SERIES, ("Series 1", "Series 2", "Series 3")))
_ex_cells = {b: int((_ex["batch"] == b).sum()) for b in ("pilot",) + T.SERIES}
_ex_t = pd.DataFrame({"Cause": list(_ex_desc), **{_ex_lab[b]: _ex_n[b].to_numpy() for b in T.SERIES}})
_ex_t["Total"] = _ex_t[list(_ex_lab.values())].sum(axis=1)
_ex_t["Description"] = [_ex_desc[c] for c in _ex_t["Cause"]]
_ex_tot = {_ex_lab[b]: int(_ex_n[b].sum()) for b in T.SERIES}
_ex_tot["Total"] = sum(_ex_tot.values())
_ex_all = sum(_ex_cells[b] for b in T.SERIES)
_ex_t = pd.concat([_ex_t, pd.DataFrame([
    {"Cause": "Total excluded", **_ex_tot, "Description": f"of {_ex_all} cells"},
    {"Cause": "Valid cells", **{_ex_lab[b]: _ex_cells[b] - _ex_tot[_ex_lab[b]] for b in T.SERIES},
     "Total": _ex_all - _ex_tot["Total"], "Description": ""}])], ignore_index=True)
_ex_valid_series = int((df["batch"].isin(T.SERIES)).sum())
assert int(_ex_t.loc[_ex_t["Cause"] == "Valid cells", "Total"].iloc[0]) == int(chain.loc["scientifically_valid", "series pooled"])
_ex_p = _ex_n["pilot"]
_ex_pilot = ", ".join(f"{int(_ex_p[c])} {c.lower()}" for c in _ex_desc if _ex_p[c])
T.register(T.write_tex(
    _ex_t, "exclusions_by_cause",
    caption="Excluded cells by cause in the three series batches.",
    label="tab:exclusions", colfmt="lrrrrl", blocks={len(_ex_desc): "Totals"},
    notes=(f"Each batch is 1200 cells; each excluded cell is counted once. Of the {_ex_all - _ex_tot['Total']} valid cells, "
           f"{_ex_valid_series} enter the analysis once the untransformed obfuscated builds, the excluded application, the excluded "
           f"release build and medtimer's packed build (coverage probes its control lacks) are set aside. The pilot, which enters no estimate, lost {int(_ex_p.sum())} of its {_ex_cells['pilot']} "
           f"cells to the same causes ({_ex_pilot}).")),
    "REPO", "5.7 Deviations and failures (exclusions)",
    "excluded cells by cause in the three series batches, with totals and valid cells; the pilot's exclusions in the note", NB)
print(_ex_t.to_string(index=False))


               Cause  Series 1  Series 2  Series 3  Total                                 Description
   Device unsuitable        40        40        40    120 one application–device pair never completed
 Application crashed        30        30        30     90          crashed repeatedly within the cell
     Scenario failed        15        15        15     45                 a scenario did not complete
Supply contamination         2         3         0      5           power reached the device mid-cell
       Harness fault         1         6         2      9             runner or power-control failure
      Total excluded        88        94        87    269                               of 3600 cells
         Valid cells      1112      1106      1113   3331                                            
